In [ ]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

VIDEO_PATH = "../data/videos/sample_warehouse.mp4"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "zone_alert_warehouse.mp4")

# Using the robust base model for the final MP4 output
print("Loading robust base model for optimal detection...")
model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

zone_polygon = np.array([
    [int(width * 0.1), int(height * 0.95)],  
    [int(width * 0.9), int(height * 0.95)],  
    [int(width * 0.9), int(height * 0.3)],   
    [int(width * 0.1), int(height * 0.3)]    
], np.int32)
zone_polygon = zone_polygon.reshape((-1, 1, 2))

print("Processing video for Safety Zone Breaches...")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.track(frame, persist=True, verbose=False)
    zone_color = (0, 255, 0)
    alert_triggered = False

    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        
        # Safely assign track IDs if available, otherwise default to -1
        if results[0].boxes.id is not None:
            track_ids = results[0].boxes.id.cpu().numpy()
        else:
            track_ids = [-1] * len(boxes)
            
        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = map(int, box)
            feet_x = int((x1 + x2) / 2)
            feet_y = y2
            
            is_inside = cv2.pointPolygonTest(zone_polygon, (float(feet_x), float(feet_y)), False)
            
            id_text = f"ID:{int(track_id)}" if track_id != -1 else "Worker"
            
            if is_inside >= 0:
                alert_triggered = True
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(frame, f"WARNING {id_text}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            else:
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                cv2.putText(frame, id_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
            
            cv2.circle(frame, (feet_x, feet_y), 5, (255, 0, 0), -1)

    if alert_triggered:
        zone_color = (0, 0, 255)
        cv2.putText(frame, "ALERT: ZONE BREACH!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    cv2.polylines(frame, [zone_polygon], isClosed=True, color=zone_color, thickness=3)
    overlay = frame.copy()
    cv2.fillPoly(overlay, [zone_polygon], zone_color)
    frame = cv2.addWeighted(overlay, 0.2, frame, 0.8, 0)

    out.write(frame)
    frame_count += 1
    if frame_count % 150 == 0:
        print(f"Processed {frame_count}/{total_frames} frames...")

cap.release()
out.release()
print(f"Zone Detection Complete! Video saved to: {OUTPUT_PATH}")

Loading robust base model for optimal detection...
Processing video for Safety Zone Breaches...
⏳ Processed 150/4548 frames...
⏳ Processed 300/4548 frames...
⏳ Processed 450/4548 frames...
⏳ Processed 600/4548 frames...
⏳ Processed 750/4548 frames...
⏳ Processed 900/4548 frames...
⏳ Processed 1050/4548 frames...
⏳ Processed 1200/4548 frames...
⏳ Processed 1350/4548 frames...
⏳ Processed 1500/4548 frames...
⏳ Processed 1650/4548 frames...
⏳ Processed 1800/4548 frames...
⏳ Processed 1950/4548 frames...
⏳ Processed 2100/4548 frames...
⏳ Processed 2250/4548 frames...
⏳ Processed 2400/4548 frames...
⏳ Processed 2550/4548 frames...
⏳ Processed 2700/4548 frames...
⏳ Processed 2850/4548 frames...
⏳ Processed 3000/4548 frames...
⏳ Processed 3150/4548 frames...
⏳ Processed 3300/4548 frames...
⏳ Processed 3450/4548 frames...
⏳ Processed 3600/4548 frames...
⏳ Processed 3750/4548 frames...
⏳ Processed 3900/4548 frames...
⏳ Processed 4050/4548 frames...
⏳ Processed 4200/4548 frames...
⏳ Processed 43